# D3-02 Parameterised characterization factors with `edges`

This notebook compares two Swiss passenger-car systems for a functional unit of **1 vehicle-kilometre**.

The main question is not “which car is universally best?” It is: **how can one mapped inventory be reevaluated when characterization factors depend on atmospheric conditions or external data?**

We first build a transparent three-gas climate example, then finish with an optional location-specific PM2.5 extension.

## Learning goals

After this notebook, you should be able to:

- load an `edges` method from a JSON asset;
- check that method rules select the intended elementary flows;
- reuse one mapped inventory while changing CH4 and N2O CFs;
- compare numeric, symbolic, and external-function CF definitions;
- explain a total score using contributions from CO2, CH4, and N2O;
- recognize the reproducibility limits of a CF based on a live web service.

## 1) Select and inspect the two vehicle systems

Both activities represent one kilometre driven by a medium EURO-6c passenger car in Switzerland. Keeping the vehicle class, emission standard, location, and unit aligned makes the fuel pathway the main difference.

In [ ]:
import json
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bw2calc as bc
import bw2data as bd
from edges import EdgeLCIA, SupplyChain

### 1.1 Find exactly one activity for each technology

Exact matching is safer than taking the first search result. If the database changes, the helper below stops with an informative error instead of silently selecting a different dataset.

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")
bafu = bd.Database("bafu")


def find_one_activity(database, name, location="CH", unit="kilometer"):
    matches = [
        activity
        for activity in database
        if activity.get("name") == name
        and activity.get("location") == location
        and activity.get("unit") == unit
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one match for {name!r}, {location!r}, {unit!r}; "
            f"found {len(matches)}."
        )
    return matches[0]


gasoline_car = find_one_activity(
    bafu,
    "Transport, passenger car, gasoline, Medium, 2018, EURO-6c",
)
compressed_gas_car = find_one_activity(
    bafu,
    "Transport, passenger car, compressed gas, Medium, 2018, EURO-6c",
)

activities = {
    "Gasoline car": gasoline_car,
    "Compressed-gas car": compressed_gas_car,
}
technology_order = list(activities)

activity_summary = pd.DataFrame(
    [
        {
            "technology": label,
            "name": activity["name"],
            "location": activity.get("location"),
            "unit": activity.get("unit"),
            "database": activity.key[0],
        }
        for label, activity in activities.items()
    ]
)
activity_summary

### 1.2 Compare direct elementary flows

One activity can contain several exchanges for the same elementary flow. We therefore **sum** matching exchanges instead of storing only one value. The logarithmic axis lets us see large CO2 flows and much smaller CH4, N2O, and PM2.5 flows together.

In [ ]:
def direct_flow_group(flow_name):
    if flow_name == "Carbon dioxide, fossil":
        return "CO2, fossil"
    if flow_name == "Carbon dioxide, non-fossil":
        return "CO2, non-fossil"
    if flow_name.startswith("Methane,"):
        return "CH4"
    if flow_name == "Dinitrogen monoxide":
        return "N2O"
    if flow_name == "Particulate Matter, < 2.5 um":
        return "PM2.5"
    return None


direct_rows = []
for label, activity in activities.items():
    for exchange in activity.biosphere():
        flow_group = direct_flow_group(exchange.input.get("name", ""))
        if flow_group:
            direct_rows.append(
                {
                    "technology": label,
                    "flow": flow_group,
                    "amount": float(exchange.get("amount")),
                }
            )

direct_flows = (
    pd.DataFrame(direct_rows)
    .groupby(["technology", "flow"], as_index=False)["amount"]
    .sum()
)
direct_flow_table = (
    direct_flows.pivot(index="technology", columns="flow", values="amount")
    .fillna(0)
    .reindex(technology_order)
)

ax = direct_flow_table.plot(kind="bar", figsize=(9, 4.5), logy=True)
ax.set_title("Direct elementary flows per vehicle-kilometre")
ax.set_xlabel("")
ax.set_ylabel("Amount [kg/km] — logarithmic scale")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Flow", frameon=False, ncol=2)
plt.tight_layout()
plt.show()

### 1.3 Load the method definitions

The three climate files characterize the same carefully selected flow names. Only the way their CF values are supplied changes. The fourth file supports the optional PM2.5 example in section 7.

In [ ]:
ASSETS_DIR = Path("tutorials/DAY 3/assets/d3-02")
if not ASSETS_DIR.exists():
    ASSETS_DIR = Path("assets/d3-02")

method_files = {
    "numeric": ASSETS_DIR / "numeric_climate_method.json",
    "symbolic": ASSETS_DIR / "symbolic_climate_method.json",
    "external": ASSETS_DIR / "external_climate_method.json",
    "external_pm_api": ASSETS_DIR / "external_pm_api_method.json",
}
methods = {
    mode: json.loads(file_path.read_text())
    for mode, file_path in method_files.items()
}

method_summary = pd.DataFrame(
    [
        {
            "mode": mode,
            "method name": methods[mode]["name"],
            "unit": methods[mode]["unit"],
            "number of rules": len(methods[mode]["exchanges"]),
        }
        for mode in method_files
    ]
)
method_summary

## 2) Build a small concentration-dependent CF model

We keep the time horizon fixed at 100 years. The following cells separate:

1. the reference atmospheric state;
2. the physical constants;
3. the equations used to calculate CH4 and N2O GWP100 values;
4. the concentration grid passed to `edges`.

This is a teaching model for parameterization, not a replacement for a complete IPCC method.

### 2.1 Reference concentrations and time horizon

In [ ]:
CH4_REFERENCE_PPB = 1866.0
N2O_REFERENCE_PPB = 332.0
H_REFERENCE_YEARS = 100.0

print(f"Reference CH4 concentration: {CH4_REFERENCE_PPB:.0f} ppb")
print(f"Reference N2O concentration: {N2O_REFERENCE_PPB:.0f} ppb")
print(f"Time horizon: {H_REFERENCE_YEARS:.0f} years")

### 2.2 Physical constants

These values follow the publication-style GWP example used by `edges`. They are kept in one cell so that assumptions are visible and easy to change.

In [ ]:
# Atmosphere and molecular masses
M_ATM = 5.15e18  # total atmospheric mass [kg]
M_AIR = 28.96    # mean molar mass of air [g/mol]
M_GAS = {"CO2": 44.01, "CH4": 16.04, "N2O": 44.013}

# Radiative-efficiency coefficients and indirect forcing
RF_COEFF = {"CH4": 0.036, "N2O": 0.12}
INDIRECT_FACTOR = {"CH4": 1.65, "N2O": 1.0}

# Atmospheric lifetimes [years]
TAU_GAS = {"CH4": 11.8, "N2O": 109.0}

# Multi-exponential impulse-response model for CO2
CO2_IRF = {
    "a0": 0.2173,
    "a": [0.2240, 0.2824, 0.2763],
    "tau": [394.4, 36.54, 4.304],
}

### 2.3 Convert radiative efficiency and integrate the response

One ppb of a gas corresponds to a gas-specific atmospheric mass. Dividing radiative forcing per ppb by that mass gives radiative forcing per kilogram.

In [ ]:
def convert_ppb_to_mass_rf(radiative_efficiency_per_ppb, gas):
    """Convert W/m2/ppb to W/m2/kg for one gas."""
    ppb_per_kg = (M_AIR / M_GAS[gas]) * (1e9 / M_ATM)
    return radiative_efficiency_per_ppb * ppb_per_kg


def agwp_co2(horizon_years):
    """Integrate the CO2 reference response over the chosen horizon."""
    response_integral = CO2_IRF["a0"] * horizon_years + sum(
        fraction * lifetime * (1 - np.exp(-horizon_years / lifetime))
        for fraction, lifetime in zip(CO2_IRF["a"], CO2_IRF["tau"])
    )
    return convert_ppb_to_mass_rf(1.37e-5, "CO2") * response_integral

In [ ]:
def radiative_efficiency_concentration(gas, concentration_ppb):
    """Apply the concentration-dependent 1/sqrt(C) relationship."""
    return (
        RF_COEFF[gas]
        / (2 * np.sqrt(concentration_ppb))
        * INDIRECT_FACTOR[gas]
    )


def agwp_gas(gas, horizon_years, concentration_ppb):
    """Integrate the response of CH4 or N2O over the time horizon."""
    mass_based_rf = convert_ppb_to_mass_rf(
        radiative_efficiency_concentration(gas, concentration_ppb),
        gas,
    )
    lifetime = TAU_GAS[gas]
    return mass_based_rf * lifetime * (
        1 - np.exp(-horizon_years / lifetime)
    )


def gwp_concentration_dependent(gas, horizon_years, concentration_ppb):
    """Return GWP by dividing the gas AGWP by the CO2 AGWP."""
    return (
        agwp_gas(gas, horizon_years, concentration_ppb)
        / agwp_co2(horizon_years)
    )

### 2.4 Calculate the reference CFs

In [ ]:
REFERENCE_CF = {
    "CH4": gwp_concentration_dependent(
        "CH4", H_REFERENCE_YEARS, CH4_REFERENCE_PPB
    ),
    "N2O": gwp_concentration_dependent(
        "N2O", H_REFERENCE_YEARS, N2O_REFERENCE_PPB
    ),
}

reference_cf_table = pd.DataFrame(
    [
        {
            "gas": gas,
            "reference CF [kg CO2-eq/kg gas]": value,
        }
        for gas, value in REFERENCE_CF.items()
    ]
)
reference_cf_table.style.format(
    {"reference CF [kg CO2-eq/kg gas]": "{:.2f}"}
)

### 2.5 Build a CH4 × N2O concentration grid

Five CH4 levels and five N2O levels give 25 atmospheric states. The vehicle inventory is unchanged across these states.

In [ ]:
ch4_levels = [1200.0, 1500.0, CH4_REFERENCE_PPB, 2200.0, 2600.0]
n2o_levels = [280.0, 310.0, N2O_REFERENCE_PPB, 360.0, 390.0]

parameter_grid = pd.DataFrame(
    [
        {
            "case": f"case_{index:02d}",
            "ch4_concentration_ppb": ch4,
            "n2o_concentration_ppb": n2o,
        }
        for index, (ch4, n2o) in enumerate(
            product(ch4_levels, n2o_levels),
            start=1,
        )
    ]
).set_index("case")

grid_parameters = {
    "concentration grid": {
        "ch4_concentration_ppb": (
            parameter_grid["ch4_concentration_ppb"].to_dict()
        ),
        "n2o_concentration_ppb": (
            parameter_grid["n2o_concentration_ppb"].to_dict()
        ),
        "ch4_reference_ppb": CH4_REFERENCE_PPB,
        "n2o_reference_ppb": N2O_REFERENCE_PPB,
        "ch4_gwp100_reference": REFERENCE_CF["CH4"],
        "n2o_gwp100_reference": REFERENCE_CF["N2O"],
        "horizon_years": H_REFERENCE_YEARS,
    }
}

print(f"Number of atmospheric cases: {len(parameter_grid)}")
print("CH4 levels [ppb]:", ch4_levels)
print("N2O levels [ppb]:", n2o_levels)

The climate JSON files use **exact elementary-flow names**:

- fossil CO2 receives a fixed CF of 1;
- fossil and biogenic CH4 share the concentration-dependent CH4 CF;
- N2O receives the concentration-dependent N2O CF.

Exact names matter. A broad rule such as `startswith("Carbon dioxide")` would also match atmospheric CO2 uptake, while `startswith("Methane")` would match substances such as halons.

## 3) Reuse one mapped inventory

Each calculation follows the same sequence:

1. `lci()` calculates the supply-chain inventory;
2. `map_exchanges()` links method rules to eligible inventory edges;
3. `evaluate_cfs()` evaluates numeric values or expressions;
4. `lcia()` calculates the characterized score.

For a parameter sweep, steps 1 and 2 run once per vehicle. Only steps 3 and 4 repeat for each atmospheric case.

In [ ]:
technology_colors = {
    "Gasoline car": "#355070",
    "Compressed-gas car": "#E56B6F",
}
contribution_colors = {
    "CO2, fossil": "#4C78A8",
    "CH4": "#F58518",
    "N2O": "#54A24B",
}

In [ ]:
def plot_score_heatmaps(results, title):
    """Plot sensitivity relative to each vehicle's reference score."""
    figure, axes = plt.subplots(
        1,
        len(technology_order),
        figsize=(11, 4.2),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    sensitivity_grids = {}

    for label in technology_order:
        score_grid = (
            results.loc[results["activity"] == label]
            .pivot(
                index="n2o_concentration_ppb",
                columns="ch4_concentration_ppb",
                values="score",
            )
            .sort_index()
            .sort_index(axis=1)
        )
        reference_score = score_grid.loc[
            N2O_REFERENCE_PPB,
            CH4_REFERENCE_PPB,
        ]
        sensitivity_grids[label] = (
            score_grid / reference_score - 1
        ) * 100

    color_limit = max(
        abs(grid.to_numpy()).max()
        for grid in sensitivity_grids.values()
    )

    for axis, label in zip(axes, technology_order):
        sensitivity_grid = sensitivity_grids[label]
        image = axis.imshow(
            sensitivity_grid,
            origin="lower",
            aspect="auto",
            cmap="RdBu_r",
            vmin=-color_limit,
            vmax=color_limit,
        )
        axis.set_title(label)
        axis.set_xlabel("CH4 concentration [ppb]")
        axis.set_xticks(range(len(sensitivity_grid.columns)))
        axis.set_xticklabels(
            [f"{value:.0f}" for value in sensitivity_grid.columns],
            rotation=35,
        )
        axis.set_yticks(range(len(sensitivity_grid.index)))
        axis.set_yticklabels(
            [f"{value:.0f}" for value in sensitivity_grid.index]
        )
        for row_index, row in enumerate(sensitivity_grid.to_numpy()):
            for column_index, change in enumerate(row):
                axis.text(
                    column_index,
                    row_index,
                    f"{change:+.1f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    axes[0].set_ylabel("N2O concentration [ppb]")
    figure.colorbar(
        image,
        ax=axes,
        label="Change from reference score [%]",
        shrink=0.86,
    )
    figure.suptitle(title)
    plt.show()

## 4) Start with a fixed numeric benchmark

This first calculation uses the reference CH4 and N2O CFs directly. Besides the total, we retain contributions by gas so that similar-looking totals can be explained.

In [ ]:
def climate_flow_group(flow_name):
    if flow_name == "Carbon dioxide, fossil":
        return "CO2, fossil"
    if flow_name.startswith("Methane,"):
        return "CH4"
    if flow_name == "Dinitrogen monoxide":
        return "N2O"
    raise ValueError(f"Unexpected characterized flow: {flow_name!r}")


numeric_rows = []
numeric_contribution_rows = []

for label, activity in activities.items():
    lca = EdgeLCIA(demand={activity: 1}, method=methods["numeric"])
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()

    numeric_rows.append(
        {
            "activity": label,
            "CH4 CF": REFERENCE_CF["CH4"],
            "N2O CF": REFERENCE_CF["N2O"],
            "score": float(lca.score),
        }
    )

    cf_table = lca.generate_cf_table(include_unmatched=False)
    for _, row in cf_table.iterrows():
        numeric_contribution_rows.append(
            {
                "activity": label,
                "gas": climate_flow_group(row["supplier name"]),
                "impact": float(row["impact"]),
            }
        )

numeric_summary = pd.DataFrame(numeric_rows)
numeric_contributions = (
    pd.DataFrame(numeric_contribution_rows)
    .groupby(["activity", "gas"], as_index=False)["impact"]
    .sum()
)

contribution_totals = numeric_contributions.groupby("activity")["impact"].sum()
for label in technology_order:
    score = numeric_summary.set_index("activity").loc[label, "score"]
    assert np.isclose(score, contribution_totals[label], rtol=1e-10)

In [ ]:
display(
    numeric_summary.style.format(
        {
            "CH4 CF": "{:.2f}",
            "N2O CF": "{:.2f}",
            "score": "{:.4f}",
        }
    )
)

contribution_table = (
    numeric_contributions.pivot(
        index="activity",
        columns="gas",
        values="impact",
    )
    .fillna(0)
    .reindex(index=technology_order, columns=["CO2, fossil", "CH4", "N2O"])
)

axis = contribution_table.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4.5),
    color=[contribution_colors[column] for column in contribution_table],
)
axis.set_title("Fixed benchmark: contribution by greenhouse gas")
axis.set_xlabel("")
axis.set_ylabel("kg CO2-eq / vehicle-km")
axis.tick_params(axis="x", rotation=0)
axis.legend(title="Characterized flow", frameon=False)

totals = contribution_table.sum(axis=1)
for position, total in enumerate(totals):
    axis.text(
        position,
        total + totals.max() * 0.015,
        f"{total:.4f}",
        ha="center",
    )
axis.set_ylim(0, totals.max() * 1.12)
plt.tight_layout()
plt.show()

relative_difference = (
    totals["Compressed-gas car"] / totals["Gasoline car"] - 1
) * 100
print(
    "Compressed-gas versus gasoline:",
    f"{relative_difference:+.1f}% for this teaching method.",
)

The two scores are not equal. With four decimal places, the gasoline and compressed-gas systems are clearly separated.

The stacked bars explain the result: compressed gas has less fossil CO2, while its larger CH4 contribution offsets part of that benefit. Remember that this is a **three-gas teaching method** and not a complete climate-change method.

In [ ]:
# Optional sanity check against a complete installed method
ipcc_method = (
    "IPCC 2021",
    "climate change",
    "global warming potential (GWP100)",
)

benchmark_rows = []
for label, activity in activities.items():
    ipcc_lca = bc.LCA({activity: 1}, ipcc_method)
    ipcc_lca.lci()
    ipcc_lca.lcia()

    custom_score = numeric_summary.set_index("activity").loc[label, "score"]
    benchmark_rows.append(
        {
            "activity": label,
            "teaching method": custom_score,
            "IPCC 2021 GWP100": float(ipcc_lca.score),
            "difference [%]": (
                custom_score / float(ipcc_lca.score) - 1
            ) * 100,
        }
    )

benchmark_check = pd.DataFrame(benchmark_rows)
benchmark_check.style.format(
    {
        "teaching method": "{:.4f}",
        "IPCC 2021 GWP100": "{:.4f}",
        "difference [%]": "{:+.2f}",
    }
)

## 5) Sweep the symbolic CH4 and N2O approximation

The symbolic JSON method rescales the reference CFs with `sqrt(C_reference / C)`. We map the inventory once per vehicle, then reevaluate the expressions for all 25 atmospheric states.

In [ ]:
symbolic_rows = []

for label, activity in activities.items():
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods["symbolic"],
        parameters=grid_parameters,
        scenario="concentration grid",
    )
    lca.lci()
    lca.map_exchanges()

    for case, values in parameter_grid.iterrows():
        lca.evaluate_cfs(scenario_idx=case)
        lca.lcia()
        symbolic_rows.append(
            {
                "case": case,
                "ch4_concentration_ppb": values["ch4_concentration_ppb"],
                "n2o_concentration_ppb": values["n2o_concentration_ppb"],
                "activity": label,
                "score": float(lca.score),
            }
        )

symbolic_results = pd.DataFrame(symbolic_rows)
print(
    f"Calculated {len(symbolic_results)} scores "
    f"({len(parameter_grid)} cases × {len(activities)} vehicles)."
)

In [ ]:
plot_score_heatmaps(
    symbolic_results,
    "Sensitivity to CH4 and N2O concentrations",
)

## Guided checkpoint: read the heatmaps

- **Predict:** Does a lower or higher gas concentration produce the larger CF?
- **Run:** Use the scaffold below to find the minimum- and maximum-score case for each vehicle.
- **Interpret:** Explain why the inventory stays fixed while the scores change.
- **Check:** Lower concentrations should give higher CFs and higher scores because the relationship is proportional to `1 / sqrt(C)`.
- **Extension:** Calculate the percentage range between the minimum and maximum case.

In [ ]:
# Exercise scaffold
# 1. Group `symbolic_results` by "activity".
# 2. Use idxmin() and idxmax() on the "score" column.
# 3. Select those rows from `symbolic_results`.

# minimum_rows = ...
# maximum_rows = ...

## 6) Evaluate the same relationship with an external Python function

The external JSON method calls `gwp_concentration_dependent(...)` instead of containing the full expression. This is useful when a CF model is too complex for a short symbolic expression.

In [ ]:
allowed_functions = {
    "gwp_concentration_dependent": gwp_concentration_dependent,
}
external_rows = []

for label, activity in activities.items():
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods["external"],
        parameters=grid_parameters,
        scenario="concentration grid",
        allowed_functions=allowed_functions,
    )
    lca.lci()
    lca.map_exchanges()

    for case, values in parameter_grid.iterrows():
        lca.evaluate_cfs(scenario_idx=case)
        lca.lcia()
        external_rows.append(
            {
                "case": case,
                "ch4_concentration_ppb": values["ch4_concentration_ppb"],
                "n2o_concentration_ppb": values["n2o_concentration_ppb"],
                "activity": label,
                "score": float(lca.score),
            }
        )

external_results = pd.DataFrame(external_rows)
print(f"Calculated {len(external_results)} external-function scores.")

In [ ]:
implementation_check = symbolic_results.merge(
    external_results,
    on=[
        "case",
        "ch4_concentration_ppb",
        "n2o_concentration_ppb",
        "activity",
    ],
    suffixes=("_symbolic", "_external"),
)
implementation_check["absolute difference"] = (
    implementation_check["score_symbolic"]
    - implementation_check["score_external"]
).abs()

maximum_difference = implementation_check["absolute difference"].max()
print(
    "Maximum symbolic–external score difference:",
    f"{maximum_difference:.3e} kg CO2-eq/km",
)
assert maximum_difference < 1e-12

### Compare score ranges across the three modes

The symbolic and external implementations should overlap because they encode the same mathematics. The numeric benchmark is a single reference point.

In [ ]:
mode_rows = []
for mode, results in [
    ("Numeric benchmark", numeric_summary),
    ("Symbolic grid", symbolic_results),
    ("External grid", external_results),
]:
    for label in technology_order:
        scores = results.loc[results["activity"] == label, "score"]
        mode_rows.append(
            {
                "mode": mode,
                "activity": label,
                "minimum": float(scores.min()),
                "maximum": float(scores.max()),
            }
        )

mode_summary = pd.DataFrame(mode_rows)

figure, axis = plt.subplots(figsize=(8.5, 4.5))
mode_order = ["Numeric benchmark", "Symbolic grid", "External grid"]
base_positions = np.arange(len(mode_order))

for offset, label in zip([-0.12, 0.12], technology_order):
    subset = (
        mode_summary.loc[mode_summary["activity"] == label]
        .set_index("mode")
        .reindex(mode_order)
    )
    midpoint = (subset["minimum"] + subset["maximum"]) / 2
    half_range = (subset["maximum"] - subset["minimum"]) / 2
    axis.errorbar(
        midpoint,
        base_positions + offset,
        xerr=half_range,
        fmt="o",
        capsize=5,
        color=technology_colors[label],
        label=label,
    )

axis.set_yticks(base_positions)
axis.set_yticklabels(mode_order)
axis.invert_yaxis()
axis.set_xlabel("kg CO2-eq / vehicle-km")
axis.set_title("Reference score and parameter-sweep ranges")
axis.legend(frameon=False)
axis.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 7) Optional extension: location-specific PM2.5 CFs from a live API

The climate example changed one CF consistently across all matched edges. This extension instead evaluates a PM2.5 CF for each matched **consumer location** and air subcompartment.

The final CF combines:

- a country-level background concentration from Open-Meteo;
- an urban, rural, or remote intake fraction;
- a simple background-dependent effect factor.

> **Important:** this section needs internet access and will change over time. It uses batched API requests for speed, but it is designed to demonstrate the `edges` interface—not to provide decision-grade PM2.5 factors.

In [ ]:
import requests

COUNTRY_CENTROIDS = {
    row["code"]: row
    for row in json.loads(
        (ASSETS_DIR / "country_centroids_iso2.json").read_text()
    )
}
OPEN_METEO_URL = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

PM25_BACKGROUND_CACHE = {}
PM25_CF_CACHE = {}

PM25_INTAKE_FRACTIONS = {
    "urban": 26e-6,
    "rural": 2.6e-6,
    "remote": 0.1e-6,
}
PM25_CATEGORY_ENVIRONMENTS = {
    ("air", "urban air close to ground"): "urban",
    ("air", "non-urban air or from high stacks"): "rural",
    ("air", "low population density, long-term"): "remote",
    ("air", "lower stratosphere + upper troposphere"): "remote",
    ("air",): "rural",
}

In [ ]:
def fetch_background_pm25_batch(country_codes, batch_size=50):
    """Fetch and cache current PM2.5 for several country centroids."""
    codes = [code.upper() for code in country_codes]
    missing_codes = [
        code for code in codes if code not in COUNTRY_CENTROIDS
    ]
    if missing_codes:
        raise ValueError(f"No centroids available for: {missing_codes}")

    pending_codes = [
        code for code in codes if code not in PM25_BACKGROUND_CACHE
    ]
    for start in range(0, len(pending_codes), batch_size):
        batch = pending_codes[start : start + batch_size]
        response = requests.get(
            OPEN_METEO_URL,
            params={
                "latitude": ",".join(
                    str(COUNTRY_CENTROIDS[code]["lat"])
                    for code in batch
                ),
                "longitude": ",".join(
                    str(COUNTRY_CENTROIDS[code]["lon"])
                    for code in batch
                ),
                "current": "pm2_5",
            },
            timeout=60,
        )
        response.raise_for_status()
        payloads = response.json()
        if isinstance(payloads, dict):
            payloads = [payloads]
        if len(payloads) != len(batch):
            raise RuntimeError(
                "Open-Meteo returned a different number of locations "
                "than requested."
            )
        for code, payload in zip(batch, payloads):
            PM25_BACKGROUND_CACHE[code] = float(
                payload["current"]["pm2_5"]
            )

    return {
        code: PM25_BACKGROUND_CACHE[code]
        for code in codes
    }


def fetch_background_pm25(country_code):
    """Fetch one value, reusing the same batch-capable cache."""
    code = country_code.upper()
    return fetch_background_pm25_batch([code])[code]


def beta_from_rr10(relative_risk_per_10ug=1.08):
    if relative_risk_per_10ug <= 1:
        raise ValueError("Relative risk must be greater than 1.")
    return relative_risk_per_10ug ** (1 / 10) - 1


def effect_factor_pm25(
    background_ug_m3,
    reference_effect_factor=54.0,
    reference_background_ug_m3=20.0,
    relative_risk_per_10ug=1.08,
):
    """Scale a reference inhaled effect factor by background PM2.5."""
    if background_ug_m3 < 0:
        raise ValueError("Background concentration must be non-negative.")
    beta = beta_from_rr10(relative_risk_per_10ug)
    scale = (
        1 + beta * reference_background_ug_m3
    ) / (1 + beta * background_ug_m3)
    return reference_effect_factor * scale

In [ ]:
def evaluate_pm25_cf(country_code, environment="rural"):
    """Return the PM2.5 CF and its intermediate components."""
    code = country_code.upper()
    environment = environment.lower()
    if environment not in PM25_INTAKE_FRACTIONS:
        raise ValueError(
            f"Environment must be one of {tuple(PM25_INTAKE_FRACTIONS)}."
        )

    cache_key = (code, environment)
    if cache_key in PM25_CF_CACHE:
        return PM25_CF_CACHE[cache_key]

    background = fetch_background_pm25(code)
    intake_fraction = PM25_INTAKE_FRACTIONS[environment]
    effect_factor = effect_factor_pm25(background)
    result = {
        "country": code,
        "country_name": COUNTRY_CENTROIDS[code]["name"],
        "environment": environment,
        "background_concentration_ug_m3": background,
        "intake_fraction_kg_inhaled_per_kg_emitted": intake_fraction,
        "effect_factor_daly_per_kg_inhaled": effect_factor,
        "cf_daly_per_kg_emitted": intake_fraction * effect_factor,
    }
    PM25_CF_CACHE[cache_key] = result
    return result


def evaluate_pm25_cf_value(country_code, environment="rural"):
    """Scalar wrapper called from the JSON method."""
    return evaluate_pm25_cf(
        country_code,
        environment,
    )["cf_daly_per_kg_emitted"]

### 7.1 Fill the API cache once

The PM method contains country-specific rows. Open-Meteo accepts several coordinate pairs in one request, so the cache is filled in small batches instead of making one request per country. A small eight-country sample is retained for the comparison plot.

In [ ]:
all_country_codes = sorted(COUNTRY_CENTROIDS)
fetch_background_pm25_batch(all_country_codes, batch_size=50)
print(
    f"Cached {len(PM25_BACKGROUND_CACHE)} country backgrounds "
    "with batched requests."
)

reference_country_codes = [
    "CH", "DE", "FR", "US", "CN", "IN", "BR", "ZA"
]
sample_country_rows = []

for code in reference_country_codes:
    for environment in ["urban", "rural", "remote"]:
        result = evaluate_pm25_cf(code, environment)
        sample_country_rows.append(
            {
                "country": result["country_name"],
                "environment": environment,
                "background PM2.5 [ug/m3]": (
                    result["background_concentration_ug_m3"]
                ),
                "CF [DALY/kg emitted]": (
                    result["cf_daly_per_kg_emitted"]
                ),
            }
        )

sample_country_reference = pd.DataFrame(sample_country_rows)

### 7.2 Run the location-specific method

The JSON method has one row per country and PM2.5 air subcompartment. `apply_strategies()` applies the location fallback cascade, while the flow `categories` choose the urban, rural, or remote intake fraction.

In [ ]:
pm_score_rows = []
pm_location_rows = []

for label, activity in activities.items():
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods["external_pm_api"],
        allowed_functions={
            "evaluate_pm25_cf_value": evaluate_pm25_cf_value
        },
    )
    lca.lci()
    lca.apply_strategies()
    lca.evaluate_cfs()
    lca.lcia()

    pm_score_rows.append(
        {
            "activity": label,
            "score": float(lca.score),
            "unit": methods["external_pm_api"]["unit"],
        }
    )

    cf_table = lca.generate_cf_table(include_unmatched=False)
    for _, row in cf_table.iterrows():
        if not str(row["supplier name"]).startswith(
            "Particulate Matter, < 2.5 um"
        ):
            continue

        categories = row.get("supplier categories", ("air",))
        categories = (
            tuple(categories)
            if isinstance(categories, (list, tuple))
            else ("air",)
        )
        pm_location_rows.append(
            {
                "activity": label,
                "consumer location": row["consumer location"],
                "environment": PM25_CATEGORY_ENVIRONMENTS.get(
                    categories,
                    "rural",
                ),
                "CF": float(row["CF"]),
                "impact": float(row["impact"]),
            }
        )

pm_score_summary = pd.DataFrame(pm_score_rows)
pm_location_results = pd.DataFrame(pm_location_rows)
display(pm_score_summary.style.format({"score": "{:.3e}"}))

### 7.3 Check the location decomposition

Before plotting, sum all location contributions and compare them with the total LCIA score. A small numerical difference is expected only from floating-point arithmetic.

In [ ]:
pm_location_summary = (
    pm_location_results.groupby(
        ["activity", "consumer location", "environment"],
        as_index=False,
    )
    .agg(CF=("CF", "mean"), impact=("impact", "sum"))
    .sort_values(["activity", "impact"], ascending=[True, False])
)

location_totals = (
    pm_location_summary.groupby("activity")["impact"].sum()
)
pm_reconciliation = pm_score_summary.set_index("activity")[["score"]]
pm_reconciliation["sum of location impacts"] = location_totals
pm_reconciliation["difference"] = (
    pm_reconciliation["score"]
    - pm_reconciliation["sum of location impacts"]
)

assert np.allclose(
    pm_reconciliation["score"],
    pm_reconciliation["sum of location impacts"],
    rtol=1e-8,
    atol=1e-15,
)
pm_reconciliation.style.format("{:.3e}")

### 7.4 Compare live CFs and supply-chain locations

The left panel shows how the environment archetype changes the CF. The right panel shows where the characterized supply-chain impacts occur for each vehicle.

In [ ]:
top_locations = (
    pm_location_summary.groupby("consumer location")["impact"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

plot_location_table = pm_location_summary.assign(
    location=lambda frame: frame["consumer location"].where(
        frame["consumer location"].isin(top_locations),
        "Other",
    )
)
plot_location_pivot = (
    plot_location_table.groupby(
        ["activity", "location"],
        as_index=False,
    )["impact"]
    .sum()
    .pivot(index="activity", columns="location", values="impact")
    .fillna(0)
    .reindex(technology_order)
)

country_cf_table = (
    sample_country_reference.pivot(
        index="country",
        columns="environment",
        values="CF [DALY/kg emitted]",
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(14, 4.8),
    constrained_layout=True,
)

country_cf_table.plot(
    kind="bar",
    ax=axes[0],
    color=["#6c757d", "#457b9d", "#d62828"],
)
axes[0].set_title("Live PM2.5 CFs")
axes[0].set_xlabel("")
axes[0].set_ylabel("DALY / kg emitted")
axes[0].tick_params(axis="x", rotation=40)
axes[0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[0].legend(title="Environment", frameon=False)

plot_location_pivot.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    colormap="tab20",
)
axes[1].set_title("PM2.5 impact by consumer location")
axes[1].set_xlabel("")
axes[1].set_ylabel("DALY / vehicle-km")
axes[1].tick_params(axis="x", rotation=0)
axes[1].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[1].legend(
    title="Location",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)
plt.show()

This extension changes two things at once: the external background data vary by country, and the intake fraction varies by air subcompartment. The result is therefore location-specific, time-sensitive, and less reproducible than the fixed concentration grid.

### 7.5 Optional supply-chain Sankey

The location chart aggregates impacts. A Sankey diagram complements it by showing the main upstream branches contributing to the gasoline-car PM2.5 result.

In [ ]:
gasoline_pm_supply_chain = SupplyChain(
    activity=gasoline_car,
    method=methods["external_pm_api"],
    amount=1,
    level=6,
    cutoff=0.01,
    cutoff_basis="total",
)

# SupplyChain creates its own EdgeLCIA instance.
gasoline_pm_supply_chain.elcia.SAFE_GLOBALS.update(
    {"evaluate_pm25_cf_value": evaluate_pm25_cf_value}
)
gasoline_pm_supply_chain.bootstrap()
gasoline_pm_sankey_df, gasoline_pm_total_score, _ = (
    gasoline_pm_supply_chain.calculate()
)

print(
    "Gasoline PM2.5 score represented by the Sankey:",
    f"{gasoline_pm_total_score:.3e} DALY/km",
)
gasoline_pm_supply_chain.plot_sankey(
    gasoline_pm_sankey_df,
    width_max=1700,
    height_max=760,
    auto_width=True,
)

## Recap

You have now:

- selected two directly comparable vehicle activities with explicit checks;
- corrected broad flow matching by using exact greenhouse-gas names;
- explained fixed climate scores with CO2, CH4, and N2O contributions;
- verified the teaching method against IPCC 2021 GWP100;
- reused mapped inventories across a 25-case parameter grid;
- shown that symbolic and external-function implementations agree;
- separated a reproducible parameter sweep from an optional live-API example;
- checked that location contributions reconcile with the PM2.5 total.

The key lesson is that dynamic characterization is auditable only when the functional unit, selected flows, parameter values, and matching rules are all visible.